<a href="https://colab.research.google.com/github/Ali-datasmith/data-science-foundations/blob/main/polars/polars_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐻 Polars for Data Science — Complete Reference Notebook

**Author:** Muhammad Ali Rajput
**Last Updated:** April 2026
**Tools:** Python 3.12 | Polars 1.35.2

---

This notebook covers Polars from the ground up — a lightning-fast
DataFrame library written in Rust, designed as a modern alternative to Pandas.

> 💡 Best viewed in **Jupyter Notebook** or **Google Colab**

---

## 📚 Table of Contents

| # | Topic | Key Methods |
|---|-------|-------------|
| 1 | Installation & Setup | `!pip install polars`, `pl.__version__` |
| 2 | Creating DataFrames | `pl.DataFrame()`, `pl.read_csv()` |
| 3 | Select & Filter | `select()`, `filter()`, `pl.col()` |
| 4 | Adding & Modifying Columns | `with_columns()`, `.alias()` |
| 5 | GroupBy & Sort | `group_by()`, `.agg()`, `sort()` |
| 6 | Join & Concat | `join()`, `concat()` |
| 7 | Lazy vs Eager Execution | `.lazy()`, `.explain()`, `.collect()` |
| 8 | Polars vs Pandas Benchmark | Performance comparison |

---

---
##To get started with `Polars` in Google Colab, you first need to install it, as it isn't pre-installed like Pandas:

In [ ]:
!pip install polars
import polars as pl

###Checking the `Polars` version

In [ ]:
print(pl.__version__)

1.35.2


---
### `read_csv` and Creating `DataFrames`
*Polars can read files or convert Python dictionaries into DataFrames. It is significantly faster than Pandas because it is written in Rust.*

In [ ]:
# Creating from a dictionary
df = pl.DataFrame({
    "product": ["Laptop", "Mouse", "Monitor"],
    "price": [1200, 25, 300],
    "sales": [10.0, 50.0, 20.0]
})

# Reading a CSV
# df = pl.read_csv("sales_data.csv")

print(df)

shape: (3, 3)
┌─────────┬───────┬───────┐
│ product ┆ price ┆ sales │
│ ---     ┆ ---   ┆ ---   │
│ str     ┆ i64   ┆ f64   │
╞═════════╪═══════╪═══════╡
│ Laptop  ┆ 1200  ┆ 10.0  │
│ Mouse   ┆ 25    ┆ 50.0  │
│ Monitor ┆ 300   ┆ 20.0  │
└─────────┴───────┴───────┘


###Explanation:
*pl.DataFrame converts dictionaries into a Polars structure. read_csv works similarly to Pandas but is optimized for multi-threaded performance.*

---
## `select` and `filter`
*In Polars, select is for choosing or transforming columns, while filter is for picking specific rows.*

In [ ]:
# Selecting specific columns
selected_df = df.select(["product", "price"])

print(selected_df)
# Filtering rows where price > 100
filtered_df = df.filter(pl.col("price") > 100)

print(filtered_df)

shape: (3, 2)
┌─────────┬───────┐
│ product ┆ price │
│ ---     ┆ ---   │
│ str     ┆ i64   │
╞═════════╪═══════╡
│ Laptop  ┆ 1200  │
│ Mouse   ┆ 25    │
│ Monitor ┆ 300   │
└─────────┴───────┘
shape: (2, 3)
┌─────────┬───────┬───────┐
│ product ┆ price ┆ sales │
│ ---     ┆ ---   ┆ ---   │
│ str     ┆ i64   ┆ f64   │
╞═════════╪═══════╪═══════╡
│ Laptop  ┆ 1200  ┆ 10.0  │
│ Monitor ┆ 300   ┆ 20.0  │
└─────────┴───────┴───────┘


###Explanation:
*select picks the vertical slices of your data. filter uses pl.col() to reference columns and apply logical conditions to rows.*

---
###`with_columns`
*This is the standard way to create new columns or modify existing ones without changing the original DataFrame structure.*

In [ ]:
# Adding a "total_revenue" column (price * sales)
df_new = df.with_columns(
    (pl.col("price") * pl.col("sales")).alias("total_revenue")
)

print(df_new)

shape: (3, 4)
┌─────────┬───────┬───────┬───────────────┐
│ product ┆ price ┆ sales ┆ total_revenue │
│ ---     ┆ ---   ┆ ---   ┆ ---           │
│ str     ┆ i64   ┆ f64   ┆ f64           │
╞═════════╪═══════╪═══════╪═══════════════╡
│ Laptop  ┆ 1200  ┆ 10.0  ┆ 12000.0       │
│ Mouse   ┆ 25    ┆ 50.0  ┆ 1250.0        │
│ Monitor ┆ 300   ┆ 20.0  ┆ 6000.0        │
└─────────┴───────┴───────┴───────────────┘


###Explanation:
*with_columns adds new data vertically. Use .alias() to name your new calculation, otherwise it will overwrite an existing column.*

---
###`group_by` and `sort`
*In Polars, grouping is used to aggregate data (like sums or averages) for specific categories, while sorting organizes the rows.*

In [ ]:
# Grouping by product and summing the sales
grouped_df = df.group_by("product").agg(
    pl.col("sales").sum().alias("total_sales")
)

# Sorting by total_sales in descending order
sorted_df = grouped_df.sort("total_sales", descending=True)

print(sorted_df)

shape: (3, 2)
┌─────────┬─────────────┐
│ product ┆ total_sales │
│ ---     ┆ ---         │
│ str     ┆ f64         │
╞═════════╪═════════════╡
│ Mouse   ┆ 50.0        │
│ Monitor ┆ 20.0        │
│ Laptop  ┆ 10.0        │
└─────────┴─────────────┘


###Explanation:
*group_by followed by .agg() combines rows based on a category. sort orders your data by a specific column in ascending or descending order.*

---
###`join` and `concat`
*join combines DataFrames horizontally based on a key, while concat stacks them vertically or horizontally.*

In [ ]:
# ensure price is Float64 in both DataFrames
df = pl.DataFrame({
    "product": ["Laptop", "Mouse", "Monitor"],
    "price": [1200.0, 25.0, 300.0], # Added .0 to make these Floats
    "sales": [10, 50, 20]
})

df_extra = pl.DataFrame({
    "product": ["Tablet"],
    "price": [500.0], # Consistent Float64 type
    "sales": [5]
})

stacked_df = pl.concat([df, df_extra])
print(stacked_df)

shape: (4, 3)
┌─────────┬────────┬───────┐
│ product ┆ price  ┆ sales │
│ ---     ┆ ---    ┆ ---   │
│ str     ┆ f64    ┆ i64   │
╞═════════╪════════╪═══════╡
│ Laptop  ┆ 1200.0 ┆ 10    │
│ Mouse   ┆ 25.0   ┆ 50    │
│ Monitor ┆ 300.0  ┆ 20    │
│ Tablet  ┆ 500.0  ┆ 5     │
└─────────┴────────┴───────┘


In [ ]:
df_inventory = pl.DataFrame({"product": ["Laptop", "Mouse"], "stock": [5, 100]})

# Joining on the "product" column
joined_df = df.join(df_inventory, on="product", how="left")
print(joined_df)

shape: (3, 4)
┌─────────┬────────┬───────┬───────┐
│ product ┆ price  ┆ sales ┆ stock │
│ ---     ┆ ---    ┆ ---   ┆ ---   │
│ str     ┆ f64    ┆ i64   ┆ i64   │
╞═════════╪════════╪═══════╪═══════╡
│ Laptop  ┆ 1200.0 ┆ 10    ┆ 5     │
│ Mouse   ┆ 25.0   ┆ 50    ┆ 100   │
│ Monitor ┆ 300.0  ┆ 20    ┆ null  │
└─────────┴────────┴───────┴───────┘


###Explanation:
*join links related data from different tables using a shared column. concat glues DataFrames together, usually by stacking rows on top of each other.*

---
###`Lazy` vs `Eager`
*Eager executes every step immediately (like Pandas). Lazy waits, plans the most efficient way to run the whole query, and only executes when you call .collect().*

In [ ]:
# Lazy execution example
lazy_plan = df.lazy().filter(pl.col("price") > 100).select("product")

# Results are only computed now
result = lazy_plan.collect()

print(result)

shape: (2, 1)
┌─────────┐
│ product │
│ ---     │
│ str     │
╞═════════╡
│ Laptop  │
│ Monitor │
└─────────┘


###Explanation:
*.lazy() creates a logical plan without moving data. .collect() triggers the execution, allowing Polars to optimize the query for speed and memory.*

---
###`Putting it all together`
*In a typical project, you will use Lazy mode to chain these operations for maximum performance before final execution.*

In [ ]:
# The "Project Style" workflow
final_result = (
    df.lazy()
    .with_columns((pl.col("price") * pl.col("sales")).alias("revenue"))
    .group_by("product")
    .agg([
        pl.col("revenue").sum(),
        pl.col("sales").sum(),  # Added this to see sales in the result
        pl.col("price").mean()  # Optional: see the average price
    ])
    .sort("revenue", descending=True)
    .collect()
)

print(final_result)

shape: (3, 4)
┌─────────┬─────────┬───────┬────────┐
│ product ┆ revenue ┆ sales ┆ price  │
│ ---     ┆ ---     ┆ ---   ┆ ---    │
│ str     ┆ f64     ┆ i64   ┆ f64    │
╞═════════╪═════════╪═══════╪════════╡
│ Laptop  ┆ 12000.0 ┆ 10    ┆ 1200.0 │
│ Monitor ┆ 6000.0  ┆ 20    ┆ 300.0  │
│ Mouse   ┆ 1250.0  ┆ 50    ┆ 25.0   │
└─────────┴─────────┴───────┴────────┘


---
###🧪Practice Set (All-Concepts)

###1. Data Loading (read_csv / DataFrame)
Create a DataFrame named df_products containing the following data:

- Product_ID: [101, 102, 103, 104, 105]

- Category: ["Electronics", "Accessories", "Electronics", "Furniture", "Accessories"]

- Base_Price: [1200.50, 25.00, 300.00, 450.00, 15.75]

- Units_Sold: [15, 120, 45, 8, 200]

In [ ]:
df_products = pl.DataFrame({
    'Product_ID' : [101, 102, 103, 104, 105],
    'Category' : ["Electronics", "Accessories", "Electronics", "Furniture", "Accessories"],
    'Base_Price' : [1200.50, 25.00, 300.00, 450.00, 15.75],
    'Units_Sold': [15, 120, 45, 8, 200]
    })
print(df_products)

shape: (5, 4)
┌────────────┬─────────────┬────────────┬────────────┐
│ Product_ID ┆ Category    ┆ Base_Price ┆ Units_Sold │
│ ---        ┆ ---         ┆ ---        ┆ ---        │
│ i64        ┆ str         ┆ f64        ┆ i64        │
╞════════════╪═════════════╪════════════╪════════════╡
│ 101        ┆ Electronics ┆ 1200.5     ┆ 15         │
│ 102        ┆ Accessories ┆ 25.0       ┆ 120        │
│ 103        ┆ Electronics ┆ 300.0      ┆ 45         │
│ 104        ┆ Furniture   ┆ 450.0      ┆ 8          │
│ 105        ┆ Accessories ┆ 15.75      ┆ 200        │
└────────────┴─────────────┴────────────┴────────────┘


###2. Selection & Filtering (select / filter)
- From df_products, create a new DataFrame that only includes the Product_ID and Base_Price for items where Units_Sold is greater than 30.

In [ ]:
df_select = df_products.filter(pl.col('Units_Sold')>30).select(['Product_ID','Base_Price'])
print(df_select)

shape: (3, 2)
┌────────────┬────────────┐
│ Product_ID ┆ Base_Price │
│ ---        ┆ ---        │
│ i64        ┆ f64        │
╞════════════╪════════════╡
│ 102        ┆ 25.0       │
│ 103        ┆ 300.0      │
│ 105        ┆ 15.75      │
└────────────┴────────────┘


###3. Feature Engineering (with_columns)
- Add a new column called Gross_Revenue to the original df_products.

- Calculation: Base_Price * Units_Sold.

- Ensure the result is rounded or kept as a float.

In [ ]:
df_products = df_products.with_columns((pl.col('Base_Price')*pl.col('Units_Sold')).alias('Gross_Reveneue'))
print(df_products)

shape: (5, 5)
┌────────────┬─────────────┬────────────┬────────────┬────────────────┐
│ Product_ID ┆ Category    ┆ Base_Price ┆ Units_Sold ┆ Gross_Reveneue │
│ ---        ┆ ---         ┆ ---        ┆ ---        ┆ ---            │
│ i64        ┆ str         ┆ f64        ┆ i64        ┆ f64            │
╞════════════╪═════════════╪════════════╪════════════╪════════════════╡
│ 101        ┆ Electronics ┆ 1200.5     ┆ 15         ┆ 18008.0        │
│ 102        ┆ Accessories ┆ 25.0       ┆ 120        ┆ 3000.0         │
│ 103        ┆ Electronics ┆ 300.0      ┆ 45         ┆ 13500.0        │
│ 104        ┆ Furniture   ┆ 450.0      ┆ 8          ┆ 3600.0         │
│ 105        ┆ Accessories ┆ 15.75      ┆ 200        ┆ 3150.0         │
└────────────┴─────────────┴────────────┴────────────┴────────────────┘


###4. Aggregation & Sorting (group_by / sort)
- Calculate the total Gross_Revenue for each Category. Sort the results so the category with the highest revenue appears at the top.

In [ ]:
df_products.group_by('Category').agg(pl.col('Gross_Reveneue').sum()).sort('Gross_Reveneue',descending=True)

Category,Gross_Reveneue
str,f64
"""Electronics""",31508.0
"""Accessories""",6150.0
"""Furniture""",3600.0


###5. Data Merging (join / concat)
- Create a second DataFrame df_discounts with columns Category and Discount_Rate ([0.1, 0.05, 0.1, 0.2]).

- Perform a left join to attach the discount rates to your main product DataFrame.

- Calculate a final Discounted_Price column.

In [ ]:
#new dataframe name df_discounts
df_discounts = pl.DataFrame({
    'Product_ID' : [101, 102, 103, 104],
    'Category' : ["Electronics", "Accessories", "Electronics", "Furniture"],
    'Discount_Rate' : [0.1, 0.05, 0.1, 0.2]
})
print(f"Last_DataFrame : \n{df_products}")
print(f"New_DataFrame : \n{df_discounts}")

Last_DataFrame : 
shape: (5, 5)
┌────────────┬─────────────┬────────────┬────────────┬────────────────┐
│ Product_ID ┆ Category    ┆ Base_Price ┆ Units_Sold ┆ Gross_Reveneue │
│ ---        ┆ ---         ┆ ---        ┆ ---        ┆ ---            │
│ i64        ┆ str         ┆ f64        ┆ i64        ┆ f64            │
╞════════════╪═════════════╪════════════╪════════════╪════════════════╡
│ 101        ┆ Electronics ┆ 1200.5     ┆ 15         ┆ 18008.0        │
│ 102        ┆ Accessories ┆ 25.0       ┆ 120        ┆ 3000.0         │
│ 103        ┆ Electronics ┆ 300.0      ┆ 45         ┆ 13500.0        │
│ 104        ┆ Furniture   ┆ 450.0      ┆ 8          ┆ 3600.0         │
│ 105        ┆ Accessories ┆ 15.75      ┆ 200        ┆ 3150.0         │
└────────────┴─────────────┴────────────┴────────────┴────────────────┘
New_DataFrame : 
shape: (4, 3)
┌────────────┬─────────────┬───────────────┐
│ Product_ID ┆ Category    ┆ Discount_Rate │
│ ---        ┆ ---         ┆ ---           │
│ i64     

In [ ]:
joined_df = df_products.join(df_discounts,on='Product_ID',how='left')
print(joined_df)

shape: (5, 7)
┌────────────┬─────────────┬────────────┬────────────┬───────────────┬──────────────┬──────────────┐
│ Product_ID ┆ Category    ┆ Base_Price ┆ Units_Sold ┆ Gross_Reveneu ┆ Category_rig ┆ Discount_Rat │
│ ---        ┆ ---         ┆ ---        ┆ ---        ┆ e             ┆ ht           ┆ e            │
│ i64        ┆ str         ┆ f64        ┆ i64        ┆ ---           ┆ ---          ┆ ---          │
│            ┆             ┆            ┆            ┆ f64           ┆ str          ┆ f64          │
╞════════════╪═════════════╪════════════╪════════════╪═══════════════╪══════════════╪══════════════╡
│ 101        ┆ Electronics ┆ 1200.5     ┆ 15         ┆ 18008.0       ┆ Electronics  ┆ 0.1          │
│ 102        ┆ Accessories ┆ 25.0       ┆ 120        ┆ 3000.0        ┆ Accessories  ┆ 0.05         │
│ 103        ┆ Electronics ┆ 300.0      ┆ 45         ┆ 13500.0       ┆ Electronics  ┆ 0.1          │
│ 104        ┆ Furniture   ┆ 450.0      ┆ 8          ┆ 3600.0        ┆ Furnit

###6. Optimization (Lazy vs Eager)
- Rewrite your entire workflow (Steps 2 through 5) using `.lazy()`. Chain all the operations together and use `.collect()` at the very end to produce the final result.

- Bonus: Use print(`df.explain()`) before the collect to see how Polars optimized your query.

In [ ]:
df_result = (
    df_products.lazy()
    .filter(pl.col('Units_Sold')>30)
    .with_columns((pl.col('Base_Price')*pl.col('Units_Sold')).alias('Gross_Revenue'))
    .group_by('Category')
    .agg(pl.col('Gross_Revenue').sum())
    .sort('Gross_Revenue',descending=True)
    .explain()
)
print(df_result)

SORT BY [descending: [true]] [col("Gross_Revenue")]
  AGGREGATE[maintain_order: false]
    [col("Gross_Revenue").sum()] BY [col("Category")]
    FROM
     WITH_COLUMNS:
     [[(col("Base_Price")) * (col("Units_Sold").cast(Float64))].alias("Gross_Revenue")] 
      FILTER [(col("Units_Sold")) > (30)]
      FROM
        DF ["Product_ID", "Category", "Base_Price", "Units_Sold", ...]; PROJECT["Category", "Base_Price", "Units_Sold"] 3/5 COLUMNS


In [ ]:
df_result = (
    df_products.lazy()
    .filter(pl.col('Units_Sold')>30)
    .with_columns((pl.col('Base_Price')*pl.col('Units_Sold')).alias('Gross_Revenue'))
    .group_by('Category')
    .agg(pl.col('Gross_Revenue').sum())
    .sort('Gross_Revenue',descending=True)
    .collect()
)
print(df_result)

shape: (2, 2)
┌─────────────┬───────────────┐
│ Category    ┆ Gross_Revenue │
│ ---         ┆ ---           │
│ str         ┆ f64           │
╞═════════════╪═══════════════╡
│ Electronics ┆ 13500.0       │
│ Accessories ┆ 6150.0        │
└─────────────┴───────────────┘


---
##`Pandas` and `Polars` Speed Comparision

In [ ]:
import time
import pandas as pd
import polars as pl
import numpy as np

# Big dataset banao
data = {"value": np.random.randint(1, 1000, 1_000_000),
        "category": np.random.choice(["A","B","C"], 1_000_000)}

# Pandas time
df_pd = pd.DataFrame(data)
start = time.time()
df_pd.groupby("category")["value"].sum()
print(f"Pandas time: {time.time()-start:.4f}s")

# Polars time
df_pl = pl.DataFrame(data)
start = time.time()
df_pl.group_by("category").agg(pl.col("value").sum())
print(f"Polars time: {time.time()-start:.4f}s")

Pandas time: 0.3712s
Polars time: 0.0642s


---

## ✅ What We Covered

| Chapter | Topic | Key Methods |
|---------|-------|-------------|
| 1 | Installation & Setup | `!pip install polars`, `pl.__version__` |
| 2 | Creating DataFrames | `pl.DataFrame()`, `pl.read_csv()` |
| 3 | Select & Filter | `select()`, `filter()`, `pl.col()` |
| 4 | Adding & Modifying Columns | `with_columns()`, `.alias()` |
| 5 | GroupBy & Sort | `group_by()`, `.agg()`, `sort()` |
| 6 | Join & Concat | `join()`, `concat()` |
| 7 | Lazy vs Eager Execution | `.lazy()`, `.explain()`, `.collect()` |
| 8 | Polars vs Pandas Benchmark | Performance comparison |

---

## 🔑 Key Takeaways

- **Polars is faster than Pandas** — written in Rust, uses all CPU cores automatically
- **Lazy execution** is Polars' superpower — it builds a query plan and optimizes before running
- **`pl.col()`** is the core of Polars expressions — everything goes through it
- **`with_columns()`** is the standard way to create or modify columns
- **`group_by().agg()`** is more explicit and faster than Pandas `groupby()`
- **`.explain()`** shows how Polars internally optimizes your query — no other library does this so cleanly

---

## 🔴 End of Polars Section

This notebook documents a complete learning journey through the Polars library —
from basic DataFrame creation to advanced lazy execution and real performance benchmarking.

*Next: Applying Polars on real-world datasets alongside Pandas for full Data Science workflows.*